<a href="https://colab.research.google.com/github/jakob-steuer/chemoinformatics_course/blob/main/chemoinformatics_8_appendix_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Appendix — asking a language model to read for you

*Optional, and nothing in the QSAR notebook depends on it. It produces no figure for your
presentation. It is here because the literature part of your project is real work, and because
knowing how these models fail is worth more than knowing how to call them.*

The task is **extractive question answering**: give a model a block of text and a question, and it
points at the span of that text which answers it — the tool you would reach for to mine ADMET
facts out of a corpus of papers.

In [ ]:
# Installs transformers and torch, and the first cell below downloads a model: a few minutes.
%pip install -q transformers torch

### One warning first

`transformers` is the standard library for this, and in **version 5.0** the
`pipeline("question-answering")` interface used by most tutorials **was removed**. Code written
against version 4 now raises an error about an unknown task.

So instead of a pipeline we write the three lines it was hiding: tokenise the question together
with the context, run the model, read off which span it points at. This works on both versions,
and it shows what "extractive" literally means — the output is a **start and end position in your
text**. The model cannot say anything that is not already there.

In [ ]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, AutoModelForSeq2SeqLM

transformers.logging.set_verbosity_error()      # the loading reports are noisy and harmless

EXTRACTIVE_MODEL = "distilbert/distilbert-base-cased-distilled-squad"
_loaded = {}


def ask(question, context, model_name=EXTRACTIVE_MODEL):
    """Extractive question answering: find the span of `context` that answers `question`."""
    if model_name not in _loaded:
        _loaded[model_name] = (AutoTokenizer.from_pretrained(model_name),
                               AutoModelForQuestionAnswering.from_pretrained(model_name))
    tokenizer, model = _loaded[model_name]

    inputs = tokenizer(question, context, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        output = model(**inputs)

    # the model scores every token for "is the answer starting / ending here?"
    start_scores = output.start_logits[0].softmax(-1)
    end_scores = output.end_logits[0].softmax(-1)
    start = int(start_scores.argmax())
    end = int(end_scores[start:].argmax()) + start

    answer = tokenizer.decode(inputs["input_ids"][0][start:end + 1], skip_special_tokens=True)
    return {"answer": answer.strip(), "score": float(start_scores[start] * end_scores[end])}


print("Extractive question answering ready.")

In [ ]:
context = """
Paracetamol, also known as acetaminophen, is a widely used over-the-counter medication.
It is well-absorbed, with an oral bioavailability of approximately 88%.
Metabolism occurs primarily in the liver, mainly through glucuronidation and sulfation.
A minor fraction is metabolized by CYP450 enzymes to a toxic intermediate, NAPQI.
At therapeutic doses, NAPQI is safely detoxified by glutathione.
However, in cases of overdose, glutathione is depleted, and NAPQI causes severe liver damage (hepatotoxicity).
"""

print(context)

Note:

This is an 'extractive' model.
It doesn't 'know' the answer, but it's an expert at finding the answer in a text.

However, imagine pointing this model at 10,000 research papers. You could ask, 'Which papers mention hERG toxicity?' in seconds."

### Exercise 1 — Predict before you run

Below are four questions. **Before running the cell**, write down for each one: what the answer
should be, and whether you expect the model to find it.

1. What is the oral bioavailability of Paracetamol?
2. Where is Paracetamol metabolized?
3. What is the main toxicity in case of overdose?
4. What is the name of the toxic intermediate?

In [ ]:
questions = [
    "What is the oral bioavailability of Paracetamol?",
    "Where is Paracetamol metabolized?",
    "What is the main toxicity in case of overdose?",
    "What is the name of the toxic intermediate?",
]

for question in questions:
    answer = ask(question, context)
    print(f"Q: {question}\nA: {answer['answer']}   (score {answer['score']:.2f})\n")

### What goes wrong, and why

Here is what we get (your scores may differ slightly with another version of the model):

| question | answer | score | |
|---|---|---|---|
| oral bioavailability? | 88 % | 0.82 | ✅ |
| **where** metabolized? | NAPQI | 0.53 | ❌ a *what*, not a *where* |
| main toxicity? | glutathione | 0.44 | ❌ that is the **antidote** |
| name of the toxic intermediate? | NAPQI | 0.97 | ✅ |

The two correct answers are the two whose wording appears almost literally in the text. The two
failures share a cause: **the model matches words that sit near each other, it does not model the
relationship between them.** "Metabolized" appears near "NAPQI", so NAPQI is offered for a *where*
question. "Toxicity" and "glutathione" appear in one sentence, so glutathione is offered — though
glutathione is what *prevents* the toxicity.

The useful part: **the confidence tracked the correctness** here, 0.82 and 0.97 against 0.53 and
0.44. Hold on to that for one more cell.

## How to Improve the Model

Here are three ways to get better answers, from a simple swap to a more advanced approach.

### 1. Use a Domain-Specific Extractive Model
This is the easiest and most direct improvement. The default model was trained on general text (news, Wikipedia). We can swap it with one trained specifically on biomedical and scientific papers.

These models will have a much better vocabulary and understanding of terms like "metabolite," "bioavailability," and "hepatotoxicity."

Examples (Hugging Face):

BioBERT: dmis-lab/biobert-base-cased-v1.1-squad (Trained on biomedical literature)

SciBERT: allenai/scibert_scivocab_cased (Trained on a broad corpus of scientific papers)

In [ ]:
for question in questions:
    answer = ask(question, context, model_name="dmis-lab/biobert-base-cased-v1.1-squad")
    print(f"Q: {question}\nA: {answer['answer']}   (score {answer['score']:.2f})\n")

### Exercise 2 — Did the domain-specific model actually help?

Compare the two tables before reading on. Would you deploy BioBERT instead of DistilBERT on this
evidence?

<details>
<summary>🔍 What we observed, and why it is a trap</summary>

| question | DistilBERT | BioBERT |
|---|---|---|
| bioavailability | 88 % (0.82) ✅ | 88 % (0.63) ✅ |
| **where** metabolized? | NAPQI (0.53) ❌ | CYP450 enzymes (0.04) ❌ |
| main toxicity? | glutathione (0.44) ❌ | **hepatotoxicity (0.14)** ✅ |
| toxic intermediate? | NAPQI (0.97) ✅ | NAPQI (0.98) ✅ |

The biomedical model fixed **one** of the two failures and still misses the "where" question. Two
conclusions, the second more important:

**1. "Use a domain-specific model" is a hypothesis, not a fix.** One fix out of four questions, on
one text, tells you nothing about which model is better. A real comparison needs a question set
with known answers and a score over all of them — which is more than many published comparisons
manage.

**2. Confidence scores are not comparable between models.** BioBERT's *correct* answer scored
**0.14**, far below the 0.7 threshold that worked for DistilBERT. A threshold tuned on one model is
meaningless on another.

</details>

### 2. Use an Abstractive Model (e.g., T5, BART)
The next step up is to use an abstractive model. Unlike extractive models, these models generate a new answer in their own words. They can synthesize information from multiple parts of the text.

This is a different task, often called text-generation or summarization, but you can frame the input to get a Q&A behavior.

How to implement (using T5):

In [ ]:
tokenizer_t5 = AutoTokenizer.from_pretrained("google/flan-t5-base")
model_t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")


def ask_generative(question, context):
    prompt = f"Answer the question using the context.\n\ncontext: {context}\n\nquestion: {question}"
    inputs = tokenizer_t5(prompt, return_tensors="pt", truncation=True, max_length=512)
    output = model_t5.generate(**inputs, max_new_tokens=40)
    return tokenizer_t5.decode(output[0], skip_special_tokens=True)


for question in questions:
    print(f"Q: {question}\nA: {ask_generative(question, context)}\n")

An abstractive model writes its own answer rather than pointing at a span, so it can phrase things
the text does not. In our run it got the bioavailability and NAPQI right and still missed both hard
questions ("CYP450 enzymes", "glutathione is depleted") — the same two failures, in nicer language.

That is the trade-off. An extractive model can only be wrong by **pointing at the wrong words**,
and it hands you the sentence so you can check it in seconds. A generative model can be wrong by
**writing something that was never in the text**, fluently and with no pointer back to a source.
For pulling numbers out of papers, the second failure mode is far more dangerous — and it is the
failure mode of every chatbot you will use for your project.

### 3. A full generative LLM

The step beyond: put the context into the prompt of a large generative model (Llama, Mistral,
Gemma) and ask it to reason. Retrieving the relevant text first and pasting it into the prompt is
**retrieval-augmented generation (RAG)**, and it is how most current literature assistants work.
It needs a GPU and more code than fits here — and, as above, it can be wrong far more fluently
than anything we ran.

### Exercise 3 — Point it at your own project

Paste the abstract of a paper about **your** drug–target pair as the context, and ask the questions
you actually need: the reported IC50 or Kd, which residues are involved, which resistance mutations
are mentioned, which assay was used.

Record two things for your presentation: **how often it is right** (check every answer against the
abstract yourself), and **where it fails** — numbers with units, relationships across two
sentences, and anything needing a comparison are the usual weak points.

In [ ]:
my_abstract = """
Paste the abstract of a paper about your target here.
"""

my_questions = [
    "What is the binding affinity?",
    "Which mutation causes resistance?",
]

for question in my_questions:
    answer = ask(question, my_abstract)
    print(f"Q: {question}\nA: {answer['answer']}   (score {answer['score']:.2f})\n")

# What to take from this

* **Extractive QA cannot answer a question the text does not answer.** It points at the most
  plausible-looking span anyway, with a confidence score attached. A high score means "this looks
  like an answer", not "this is true".
* **Abstractive models phrase better and invent more.** They write their own sentence, so they can
  write one the paper does not support.
* **Every number on a slide needs the sentence it came from.** That is the rule for the project:
  use any tool you like, and be able to defend each figure without it.